# Dunnhumby anchored M2 v1.4 체크포인트 진단 (재학습 없음)

v1.4 `joint_nv_anchored` seed 42 validation 체크포인트를 읽어 ID/N/V 블록과 N/V 점수 강도 0~1배를 평가합니다. 학습·test·holdout은 실행하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = 'b6e49ed628bee2f22b9e515367b7de7309f169ee'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)
], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)


In [ ]:
import importlib, json, sys, torch
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', '') or ''
    if module_path and str(repo) in str(module_path):
        del sys.modules[module_name]
importlib.invalidate_caches()

from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_anchored_dunnhumby_run,
    preflight_summary,
    run_checkpoint_diagnostics,
)
assert torch.cuda.is_available(), (
    '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
cfg = configure_anchored_dunnhumby_run()
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['code_version'] == 'm2-joint-nv-lightgcn-v1.4'
assert summary['dataset'] == 'dunnhumby'
assert summary['seed'] == 42
assert summary['models'] == ['m1', 'joint_nv_anchored']
assert summary['gamma']['initial_score_strength'] == 0.1
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
assert summary['out_dir'].endswith('_m2_joint_nv_anchored_v14')
print('설정 확인 완료. 다음 셀은 기존 체크포인트를 읽어 validation 평가만 합니다.')


In [ ]:
MULTIPLIERS = (0.0, 0.125, 0.25, 0.375, 0.5, 0.75, 1.0)
diagnostic_df = run_checkpoint_diagnostics(
    cfg,
    strength_multipliers=MULTIPLIERS,
)
assert diagnostic_df.attrs['training_performed'] is False


In [ ]:
metrics = [
    'view', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10', 'eff_catalog@10',
]
print('===== 1. ID/N/V 블록 분해 =====')
display(diagnostic_df[[c for c in metrics if c in diagnostic_df.columns]])

comparison = diagnostic_df.attrs['block_comparison']
focus = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
]
print('===== 2. M1 대비 ID 회복과 N/V 증분 =====')
display(comparison[comparison['metric'].isin(focus)].reset_index(drop=True))

print('===== 3. 블록별 실효강도 =====')
print(json.dumps(diagnostic_df.attrs['block_score_summary'], ensure_ascii=False, indent=2))

curve = diagnostic_df.attrs['strength_curve'].copy()
m1 = diagnostic_df.attrs['external_m1']
accuracy = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50']
for metric in accuracy:
    curve[f'{metric}_ratio_vs_m1'] = curve[metric] / float(m1[metric])
curve['accuracy_6_guard_pass'] = curve[[f'{m}_ratio_vs_m1' for m in accuracy]].min(axis=1) >= 0.99
curve['revenue_delta_vs_m1'] = curve['revenue@10'] - float(m1['revenue@10'])
curve['joint_pass'] = curve['accuracy_6_guard_pass'] & (curve['revenue_delta_vs_m1'] > 0)
print('===== 4. v1.4 N/V 점수 0~1배 강도 곡선 =====')
display(curve[[c for c in [
    'nv_score_multiplier', 'recall@10', 'ndcg@10', 'recall@20',
    'ndcg@20', 'recall@50', 'ndcg@50', 'revenue@10',
    'revenue_delta_vs_m1', 'coverage@10', 'n_distinct@10',
    'accuracy_6_guard_pass', 'joint_pass'
] if c in curve.columns]])

eligible = curve[curve['joint_pass']].sort_values(
    ['revenue@10', 'nv_score_multiplier'], ascending=[False, True]
)
if len(eligible):
    print('과개입 가설 지지: M1 정확도를 보호하면서 경제지표가 증가하는 축소점이 있습니다.')
    display(eligible.head(1))
else:
    print('강도 축소만으로는 M1을 넘지 못했습니다. ID-only와 ID+N/ID+V를 보고 anchor 또는 학습방향 문제로 판독합니다.')
print('결과 파일:', diagnostic_df.attrs['result_paths'])
print('완료. 1~4 출력을 그대로 공유해 주세요.')
